# Link Prediction -- GraphSAGE

Structurally parallel to `link_prediction_rgcn.ipynb`: a small validation
search over hidden dimension only (`DIM_GRID`), with everything else
anchored on this model+schema's already-tuned node-classification
hyperparameters from `graphsage/{schema}_sequential_tuning_results.csv` --
same `get_anchor_hparams` pattern RGCN's notebook uses, not the original
GraphSAGE paper's fixed literature setup this notebook used previously.
Swaps in `GraphSAGEEncoder` -- still paired with the same `DistMultDecoder`,
since a decoder doesn't care which encoder produced the embeddings.

**Run `link_prediction_setup.ipynb` first.**

In [ ]:
%restart_python

In [ ]:
# Mitigates fragmentation and clears GPU memory a leftover process may hold.
# Must run before any CUDA context is created, right after the restart above.
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

In [ ]:
import os
import sys

repo_path = '/Workspace/Users/dversch@emea.corpdir.net/.bundle/KG_Model/prod/files'
if repo_path not in sys.path:
    sys.path.append(repo_path)

import torch
import pandas as pd

from node_prediction.gnn_tools import cleanup, parse_neighbors
from link_prediction.link_prediction_tools import GraphSAGEEncoder, DistMultDecoder, run_small_lp_validation_search, run_small_lp_secondary_search, quick_sanity_check

In [ ]:
volume = "/Volumes/westeurope_cac_dwh_prd/ds_ccc/ds-landing/sulo_km/"
LP_DIR = f"{volume}link_prediction/"

FEATURE_DIM = 1024

# Deliberately below Hamilton et al. (2017)'s default hidden dim (256) --
# compute-constrained, same reasoning as link_prediction_rgcn.ipynb's DIM_GRID.
DIM_GRID = [32, 64]

# Run this notebook once per schema, in separate kernel sessions -- see
# link_prediction_setup.ipynb's SCHEMA_LABEL convention.
SCHEMA_LABEL = 'sulo'  # <<< CHANGE TO 'base' FOR THE SECOND RUN

# True resumes from the last checkpoint per hidden_dim instead of
# retraining from scratch (see train_link_prediction_model's save_path/resume).
RESUME_LP_TRAINING = False
CHECKPOINT_DIR = f"{LP_DIR}checkpoints/"

# Caps triples ranked per periodic eval checkpoint, not the final test evaluation.
MAX_VAL_EVAL_TRIPLES = 1000


def get_anchor_hparams(tuning_csv_path):
    """Anchors everything except dim on the NC-tuned winner (peak val_roc_auc,
    Phase 3 verification). num_bases is read for parity with RGCN's anchor
    dict but unused -- SAGEConv has no relation-basis concept."""
    df = pd.read_csv(tuning_csv_path)
    ok = df[df['status'] == 'ok']
    phase3 = ok[ok['phase'].isin(['3_Verification', '3_Verification_Cached'])]
    if len(phase3) == 0:
        phase3 = ok
    best = phase3.loc[phase3['val_roc_auc'].idxmax()]
    bases = best['num_bases']
    bases = None if pd.isna(bases) or bases == 'None' else int(float(bases))
    return {
        'lr': float(best['lr']),
        'weight_decay': float(best['weight_decay']),
        'num_bases': bases,  # unused by GraphSAGEEncoder -- see docstring above
        'layers': int(best['layers']),
        'neighbors': parse_neighbors(best['neighbors']),
        'epochs': 100,
        'patience': 15,
        'eval_frequency': 5,  # re-embeds every relation's full candidate pool -- too costly every epoch
        # neg_per_pos=4: 1-vs-1 training contrast is easier than the full-pool
        # eval task (see link_prediction_rgcn.ipynb's identical setting).
        'neg_per_pos': 4,
        # 'macro' (unweighted per-relation MRR) instead of 'pooled', which is
        # dominated by huge pools like Task -- keeps early stopping sensitive
        # to smaller, more learnable relations.
        'monitor': 'macro',
    }

## Run the small validation search + final test evaluation

In [ ]:
lp_data = torch.load(f"{LP_DIR}{SCHEMA_LABEL}_lp_data.pt", weights_only=False)
# mp_sub_graph_data (not sub_graph_data): has the disjoint train-supervision
# slice removed so training never sees the edge it's predicting as a neighbor
# (see link_prediction_setup.ipynb's "Mask held-out triples" section).
sub_graph_data = lp_data['mp_sub_graph_data']
# vector_table kept on CPU -- encoders gather from it per-batch and move
# only the small result to GPU (see link_prediction_tools._FeaturizationMixin).
vector_table = lp_data['vector_table']

node_index = sub_graph_data.n_id_to_vec_row
usage_counts = torch.bincount(node_index, minlength=vector_table.shape[0])
categorical_ids = torch.where(usage_counts > 1)[0]

nc_tuning_csv = f"{volume}graphsage/{SCHEMA_LABEL}_sequential_tuning_results.csv"
anchor = get_anchor_hparams(nc_tuning_csv)
print(f"[{SCHEMA_LABEL}] Anchored hyperparameters from NC tuning: {anchor}")

def encoder_factory(dim, _vt=vector_table, _cat=categorical_ids, _layers=anchor['layers']):
    return GraphSAGEEncoder(
        in_channels=FEATURE_DIM + 1,
        hidden_channels_list=[dim] * _layers,
        vector_table=_vt,
        categorical_row_ids=_cat,
    )

def decoder_factory(dim, _nr=lp_data['decoder_num_relations']):
    return DistMultDecoder(num_relations=_nr, hidden_dim=dim)

final_encoder, final_decoder, test_metrics, search_df, best_dim = run_small_lp_validation_search(
    model_name='GraphSAGE', encoder_factory=encoder_factory, decoder_factory=decoder_factory,
    sub_graph_data=sub_graph_data,
    train_triples_by_relation=lp_data['train_sup_triples_by_relation'],
    val_triples_by_relation=lp_data['val_triples_by_relation'],
    test_triples_by_relation=lp_data['test_triples_by_relation'],
    candidate_pools=lp_data['candidate_pools'],
    true_tails_by_relation=lp_data['true_tails_by_relation'],
    relation_fwd_ids=lp_data['relation_fwd_ids'],
    dim_grid=DIM_GRID,
    fixed_hparams=anchor,
    csv_path=f"{LP_DIR}graphsage_{SCHEMA_LABEL}_lp_dimsearch_tuning_results.csv",
    schema_label=SCHEMA_LABEL,
    checkpoint_dir=CHECKPOINT_DIR,
    resume=RESUME_LP_TRAINING,
    max_val_eval_triples=MAX_VAL_EVAL_TRIPLES,
)

rows = []
for relation_name, m in test_metrics.items():
    # reflects only the winning config, not every dim tried
    row = {'model': 'GraphSAGE', 'schema': SCHEMA_LABEL, 'hidden_dim': best_dim, 'relation': relation_name}
    if relation_name in lp_data['train_sup_triples_by_relation']:
        # SUPERVISED (disjoint) slice, not the full pre-split train count
        row['num_train_triples'] = int(lp_data['train_sup_triples_by_relation'][relation_name].size(0))
        row['num_val_triples'] = int(lp_data['val_triples_by_relation'][relation_name].size(0))
    row.update(m)
    rows.append(row)
results_df = pd.DataFrame(rows)
dim_results_path = f"{LP_DIR}graphsage_{SCHEMA_LABEL}_lp_dimsearch_results.csv"
results_df.to_csv(dim_results_path, index=False)
print(f"\nSaved (dim search only, superseded by the neg_per_pos step below): {dim_results_path}")
print(results_df.to_string(index=False))

del final_encoder, final_decoder
cleanup()

## Sequential LP tuning extension: neg_per_pos

Additive on top of the dim search -- `dim` locked, only `neg_per_pos`
swept (GraphSAGE has no bases step, unlike RGCN). This step's output is
the notebook's canonical results file.

In [ ]:
NEG_PER_POS_GRID = [1, 4, 10]

neg_csv_path = f"{LP_DIR}graphsage_{SCHEMA_LABEL}_lp_tuning_results.csv"
final_encoder, final_decoder, test_metrics, neg_search_df, best_neg_per_pos = run_small_lp_secondary_search(
    model_name='GraphSAGE', param_name='neg_per_pos', param_grid=NEG_PER_POS_GRID, dim=best_dim,
    encoder_factory=encoder_factory, decoder_factory=decoder_factory,
    sub_graph_data=sub_graph_data,
    train_triples_by_relation=lp_data['train_sup_triples_by_relation'],
    val_triples_by_relation=lp_data['val_triples_by_relation'],
    test_triples_by_relation=lp_data['test_triples_by_relation'],
    candidate_pools=lp_data['candidate_pools'],
    true_tails_by_relation=lp_data['true_tails_by_relation'],
    relation_fwd_ids=lp_data['relation_fwd_ids'],
    fixed_hparams=anchor,
    csv_path=neg_csv_path,
    schema_label=SCHEMA_LABEL,
    checkpoint_dir=CHECKPOINT_DIR,
    resume=RESUME_LP_TRAINING,
    max_val_eval_triples=MAX_VAL_EVAL_TRIPLES,
    rebuild_encoder=False,
)
history_path = neg_csv_path.replace('.csv', '_training_history.csv')
print(f"[{SCHEMA_LABEL}] Best neg_per_pos: {best_neg_per_pos}")

rows = []
for relation_name, m in test_metrics.items():
    row = {'model': 'GraphSAGE', 'schema': SCHEMA_LABEL, 'hidden_dim': best_dim,
           'neg_per_pos': best_neg_per_pos, 'relation': relation_name}
    if relation_name in lp_data['train_sup_triples_by_relation']:
        row['num_train_triples'] = int(lp_data['train_sup_triples_by_relation'][relation_name].size(0))
        row['num_val_triples'] = int(lp_data['val_triples_by_relation'][relation_name].size(0))
    row.update(m)
    rows.append(row)
results_df = pd.DataFrame(rows)
results_path = f"{LP_DIR}graphsage_{SCHEMA_LABEL}_lp_results.csv"
results_df.to_csv(results_path, index=False)
print(f"\nSaved final results (dim={best_dim}, neg_per_pos={best_neg_per_pos}): {results_path}")
print(results_df.to_string(index=False))

del final_encoder, final_decoder
cleanup()

## Quick sanity check

In [ ]:
# history_path/results_path are set by the neg_per_pos sequential-search
# cell above (the final, canonical output of the dim -> neg_per_pos chain)
_ = quick_sanity_check(results_path, history_path, 'GraphSAGE', SCHEMA_LABEL, 'hidden_dim', best_dim)